# Notebook 09 — Test-Retest Reliability & Cross-Measure Correlations

## Test-retest reliability
Uses the Haddara (2022) longitudinal dataset (70 subjects × 7 days). Days 2–7 give 15 unique day-pairs. For each pair and each measure, we compute both Pearson *r* and the Intraclass Correlation Coefficient (ICC, two-way random, single measures — ICC(2,1)).

## Cross-measure correlations
Pearson correlation matrix across all 20 measures, computed for Haddara, Maniscalco, and Shekhar. Shows which measures carry redundant information.

In [1]:
import matplotlib
matplotlib.use('Agg')
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── locate repository root and add src/ to path ──────────────
REPO = os.path.abspath(os.path.join(
    os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
DATA = os.path.join(REPO, 'matlab', 'metasignal_mat', 'Preprocess', 'orig_csv_files')
OUT  = os.path.join(REPO, 'notebooks', 'precomputed')
sys.path.insert(0, os.path.join(REPO, 'src'))
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
from scipy import stats
from metasignal.stdpy.compute_all import compute_all_measures
print("metasignal loaded successfully.")


metasignal loaded successfully.


In [2]:
MEASURE_NAMES = [
    "meta-d'", "AUC2", "Gamma", "Phi", "DeltaConf",
    "M-Ratio", "AUC2-Ratio", "Gamma-Ratio", "Phi-Ratio", "DeltaConf-Ratio",
    "M-Diff",  "AUC2-Diff",  "Gamma-Diff",  "Phi-Diff",  "DeltaConf-Diff",
    "meta-noise", "meta-uncertainty",
    "d'", "Criterion", "Confidence",
]
N_MEAS = 20


In [3]:
# ── t-test matching MATLAB perform_ttest.m ───────────────────
def ttest_1samp(data):
    """One-sample t-test vs 0. Returns (t, df, p, cohen_d, ci_lo, ci_hi).
    Cohen's d = t/sqrt(n) matching MATLAB: Cohen_d = t/sqrt(df+1)."""
    x = np.asarray(data, float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n < 2:
        return (np.nan,)*6
    t, p = stats.ttest_1samp(x, 0)
    d    = t / np.sqrt(n)
    sem  = x.std(ddof=1) / np.sqrt(n)
    ci   = x.mean() + stats.t.ppf([0.025, 0.975], n-1) * sem
    return t, n-1, p, d, ci[0], ci[1]

def p_stars(p):
    if np.isnan(p): return ''
    if p < 0.001:   return '***'
    if p < 0.01:    return '**'
    if p < 0.05:    return '*'
    return 'ns'

# ── one-way repeated-measures ANOVA ─────────────────────────
def rm_anova_1way(data_2d):
    """data_2d: (n_subjects, n_conditions). Returns (F, df_b, df_e, p, eta2_p)."""
    n, k   = data_2d.shape
    grand  = np.nanmean(data_2d)
    row_m  = np.nanmean(data_2d, axis=1, keepdims=True)
    col_m  = np.nanmean(data_2d, axis=0, keepdims=True)
    ss_b   = n * np.sum((col_m - grand)**2)
    ss_s   = k * np.sum((row_m - grand)**2)
    ss_e   = np.sum((data_2d - grand)**2) - ss_b - ss_s
    df_b, df_e = k-1, (n-1)*(k-1)
    F  = (ss_b/df_b) / (ss_e/df_e)
    p  = stats.f.sf(F, df_b, df_e)
    return F, df_b, df_e, p, ss_b/(ss_b+ss_e)

# ── 3-SD outlier removal per level per measure ───────────────
def remove_3sd_outliers(arr):
    """arr: (n_sub, n_levels, n_meas). Matches MATLAB ana_taskPerformance.m."""
    out = arr.copy()
    _, n_lev, n_meas = out.shape
    for m in range(n_meas):
        for lv in range(n_lev):
            col = out[:, lv, m]
            mu, sd = np.nanmean(col), np.nanstd(col, ddof=1)
            if not np.isnan(mu) and sd > 0:
                out[(col < mu-3*sd) | (col > mu+3*sd), lv, m] = np.nan
        bad = np.isnan(out[:, :, m]).any(axis=1)
        out[bad, :, m] = np.nan
    return out

print("Statistical helper functions defined.")


Statistical helper functions defined.


In [4]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

## Load data

In [5]:
ACC_LO, ACC_HI, MAX_PROP = 0.60, 0.95, 0.85
# ── exclusion thresholds (match MATLAB step2_preprocessData.m) ──
ACC_LO, ACC_HI, MAX_PROP = 0.60, 0.95, 0.85

def _exclude(acc, pr, pc):
    """Return True if subject should be excluded."""
    return acc < ACC_LO or acc > ACC_HI or pr > MAX_PROP or pc > MAX_PROP

def preprocess_haddara():
    """Load Haddara 2022 Expt2. n_ratings=4. Returns list of subject dicts."""
    df = pd.read_csv(os.path.join(DATA, 'data_Haddara_2022_Expt2.csv'))
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        g = g.dropna(subset=['Stimulus','Response','Confidence'])
        stim = g['Stimulus'].to_numpy(float)
        resp = g['Response'].to_numpy(float)
        conf = g['Confidence'].to_numpy(float)
        day  = g['Day'].to_numpy(float)
        acc = np.mean(stim == resp)
        pr  = np.max(np.unique(resp, return_counts=True)[1]) / len(resp)
        pc  = np.max(np.unique(conf, return_counts=True)[1]) / len(conf)
        if _exclude(acc, pr, pc): continue
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf, day=day, n_ratings=4))
    return subjects

def preprocess_maniscalco():
    """Load Maniscalco 2017 Expt1. NaN responses counted as incorrect. n_ratings=4."""
    df = pd.read_csv(os.path.join(DATA, 'data_Maniscalco_2017_expt1.csv'))
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        stim_all = g['Stimulus'].to_numpy(float)
        resp_all = g['Response'].to_numpy(float)
        conf_all = g['Confidence'].to_numpy(float)
        # MATLAB: NaN responses count as incorrect
        correct = np.where(np.isnan(resp_all), 0.0, (stim_all == resp_all).astype(float))
        acc = np.mean(correct)
        pr  = np.max(np.bincount(resp_all[~np.isnan(resp_all)].astype(int))) / len(resp_all)
        valid_c = conf_all[~np.isnan(conf_all)]
        pc  = np.max(np.bincount(valid_c.astype(int))) / len(conf_all)
        if _exclude(acc, pr, pc): continue
        ok = ~(np.isnan(stim_all) | np.isnan(resp_all) | np.isnan(conf_all))
        subjects.append(dict(sid=sid, stim=stim_all[ok], resp=resp_all[ok],
                             conf=conf_all[ok], n_ratings=4))
    return subjects

def preprocess_shekhar():
    """Load Shekhar 2021. Continuous conf (50–100) binned to n_ratings=6 levels."""
    df = pd.read_csv(os.path.join(DATA, 'data_Shekhar_2021.csv'))
    n_ratings = 6
    edges = np.linspace(50, 100, n_ratings + 1)
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        stim     = g['Stimulus'].to_numpy(float)
        resp     = g['Response'].to_numpy(float)
        conf_raw = g['Confidence'].to_numpy(float)
        contrast = g['Contrast'].to_numpy(float)
        conf = np.clip(np.digitize(conf_raw, edges, right=True), 1, n_ratings)
        acc = np.mean(stim == resp)
        pr  = np.max(np.unique(resp, return_counts=True)[1]) / len(resp)
        pc  = np.max(np.unique(conf, return_counts=True)[1]) / len(conf)
        if _exclude(acc, pr, pc): continue
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf,
                             contrast=contrast, n_ratings=n_ratings))
    return subjects

def preprocess_rouault(expt=1):
    """Load Rouault 2018 Expt1 or Expt2. Stereotypy on raw conf; transform after. n_ratings=6."""
    df = pd.read_csv(os.path.join(DATA, f'data_Rouault_2018_Expt{expt}.csv'))
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        g = g.dropna(subset=['Stimulus','Response','Confidence'])
        stim     = g['Stimulus'].to_numpy(float)
        resp     = g['Response'].to_numpy(float)
        conf_raw = g['Confidence'].to_numpy(float)
        dotdiff  = g['DotDiff'].to_numpy(float)
        acc = np.mean(stim == resp)
        pr  = np.max(np.bincount(resp.astype(int))) / len(resp)
        # MATLAB: stereotypy on RAW conf (1-11) BEFORE transformation
        pc  = np.max(np.bincount(conf_raw.astype(int))) / len(conf_raw)
        if _exclude(acc, pr, pc): continue
        conf = np.clip(conf_raw - 5, 1, None) if expt == 1 else conf_raw.copy()
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf,
                             contrast=dotdiff, n_ratings=6))
    return subjects

def preprocess_locke():
    """Load Locke 2020 (test trials only). n_ratings=2."""
    df = pd.read_csv(os.path.join(DATA, 'data_Locke_2020.csv'))
    df = df[df['Training'] == 0].copy()
    df['Confidence'] = df['Confidence'] + 1   # 0/1 → 1/2
    subjects = []
    for sid, g in df.groupby('Subj_idx'):
        g = g.dropna(subset=['Stimulus','Response','Confidence'])
        stim      = g['Stimulus'].to_numpy(float)
        resp      = g['Response'].to_numpy(float)
        conf      = g['Confidence'].to_numpy(float)
        condition = g['Condition'].to_numpy(float)
        acc = np.mean(stim == resp)
        pr  = np.max(np.unique(resp, return_counts=True)[1]) / len(resp)
        if acc < ACC_LO or acc > ACC_HI or pr > MAX_PROP: continue
        subjects.append(dict(sid=sid, stim=stim, resp=resp, conf=conf,
                             condition=condition, n_ratings=2))
    return subjects

print("Preprocessing functions defined.")

ha_raw  = np.load(os.path.join(OUT, 'haddara_mle.npz'))['raw']   # (70,20)
ma_raw  = np.load(os.path.join(OUT, 'maniscalco_mle.npz'))['raw']
sh_diff = np.load(os.path.join(OUT, 'shekhar_mle.npz'))['diff']
sh_raw  = np.nanmean(sh_diff, axis=1)   # (20,20) averaged over contrasts
ha      = preprocess_haddara()
days_all = sorted(set(int(d) for s in ha for d in np.unique(s['day'])))
print(f"Haddara: {len(ha)} subjects, {len(days_all)} unique day-values")
days_unique = np.unique(np.concatenate([np.unique(s['day']) for s in ha]))
print("Days available:", days_unique)


Preprocessing functions defined.


Haddara: 70 subjects, 7 unique day-values
Days available: [1. 2. 3. 4. 5. 6. 7.]


## Compute measures per day (days 2–7)

In [6]:
DAYS = [2,3,4,5,6,7]
TT_PATH = os.path.join(OUT, 'haddara_testRetest.npz')
tt = None
if os.path.exists(TT_PATH):
    tt = np.load(TT_PATH)['data']   # (70, 6, 20)
    print("Loaded test-retest data:", tt.shape)
else:
    print("Test-retest cache not found.")
    print("Requires ~14 min (70 subs x 6 days x MLE). Run precompute_test_retest.py to generate cache.")
    print("Skipping ICC computation — showing MATLAB reference values instead.")


Loaded test-retest data: (70, 6, 20)


## ICC(2,1) computation

Two-way random effects ICC for single measures (Shrout & Fleiss, 1979):  
$ICC = \frac{MS_R - MS_E}{MS_R + (k-1)MS_E + k(MS_C - MS_E)/n}$

where $MS_R$ = mean square rows, $MS_C$ = mean square columns, $MS_E$ = mean square error.

In [7]:
if tt is not None:
    def icc_21(data_2d):
        """ICC(2,1) — two-way random, single rater. data_2d: (n_subjects, n_raters)."""
        n, k = data_2d.shape
        grand  = np.nanmean(data_2d)
        row_m  = np.nanmean(data_2d, axis=1)
        col_m  = np.nanmean(data_2d, axis=0)
        ss_r   = k * np.sum((row_m - grand)**2)
        ss_c   = n * np.sum((col_m - grand)**2)
        ss_e   = np.sum((data_2d - row_m[:,None] - col_m[None,:] + grand)**2)
        ms_r   = ss_r / (n-1)
        ms_c   = ss_c / (k-1)
        ms_e   = ss_e / ((n-1)*(k-1))
        icc    = (ms_r - ms_e) / (ms_r + (k-1)*ms_e + k*(ms_c-ms_e)/n)
        return float(icc)
    
    # Average over all 15 day-pairs
    from itertools import combinations
    day_pairs = list(combinations(range(len(DAYS)), 2))
    
    print(f"{'Measure':<20} {'Mean ICC':>10} {'Mean r':>8}")
    print("-"*42)
    icc_avgs, r_avgs = [], []
    for m, name in enumerate(MEASURE_NAMES):
        iccs, rs = [], []
        for di, dj in day_pairs:
            x, y = tt[:, di, m], tt[:, dj, m]
            ok   = ~np.isnan(x) & ~np.isnan(y)
            if ok.sum() < 5: continue
            data_pair = np.column_stack([x[ok], y[ok]])
            iccs.append(icc_21(data_pair))
            r, _ = pearsonr(x[ok], y[ok])
            rs.append(r)
        avg_icc = np.nanmean(iccs) if iccs else np.nan
        avg_r   = np.nanmean(rs)   if rs   else np.nan
        icc_avgs.append(avg_icc); r_avgs.append(avg_r)
        fmt = lambda v: f"{v:.3f}" if not np.isnan(v) else "  NaN"
        print(f"{name:<20} {fmt(avg_icc):>10} {fmt(avg_r):>8}")
    print(f"\nMean ICC (non-NaN): {np.nanmean(icc_avgs):.3f}")
    print(f"Mean r   (non-NaN): {np.nanmean(r_avgs):.3f}")
    


Measure                Mean ICC   Mean r
------------------------------------------
meta-d'                   0.776    0.782
AUC2                      0.778    0.778
Gamma                     0.750    0.763
Phi                       0.712    0.718
DeltaConf                 0.826    0.828
M-Ratio                   0.552    0.562
AUC2-Ratio                0.497    0.502
Gamma-Ratio               0.163    0.205
Phi-Ratio                 0.126    0.129
DeltaConf-Ratio           0.150    0.156
M-Diff                    0.658    0.671
AUC2-Diff                 0.481    0.489
Gamma-Diff                0.497    0.507
Phi-Diff                  0.491    0.497
DeltaConf-Diff            0.451    0.461
meta-noise                0.566    0.577
meta-uncertainty          0.285    0.321
d'                        0.839    0.858
Criterion                 0.809    0.829
Confidence                0.934    0.939

Mean ICC (non-NaN): 0.567
Mean r   (non-NaN): 0.579


## Cross-measure correlations

In [8]:
FIGS = os.path.join(REPO, 'notebooks', 'figures')
os.makedirs(FIGS, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (raw, label) in zip(axes, [
    (ha_raw, 'Haddara (n=70)'),
    (ma_raw, 'Maniscalco (n=22)'),
    (sh_raw, 'Shekhar (n=20)'),
]):
    R = np.full((N_MEAS, N_MEAS), np.nan)
    for i in range(N_MEAS):
        for j in range(N_MEAS):
            xi, xj = raw[:, i], raw[:, j]
            ok = ~np.isnan(xi) & ~np.isnan(xj)
            if ok.sum() >= 3:
                r, _ = pearsonr(xi[ok], xj[ok])
                R[i, j] = r
    im = ax.imshow(R, vmin=-1, vmax=1, cmap='RdYlBu_r', aspect='auto')
    ax.set_title(label, fontweight='bold')
    ax.set_xticks(range(N_MEAS)); ax.set_yticks(range(N_MEAS))
    ax.set_xticklabels(MEASURE_NAMES, rotation=90, fontsize=6)
    ax.set_yticklabels(MEASURE_NAMES, fontsize=6)
    plt.colorbar(im, ax=ax, shrink=0.7)
    # Mean off-diagonal correlation
    mask = ~np.eye(N_MEAS, dtype=bool) & ~np.isnan(R)
    print(f"{label}: mean |r| = {np.nanmean(np.abs(R[mask])):.3f}")
plt.suptitle('Cross-Measure Correlations (Pearson r)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'fig_correlations.png'), dpi=150, bbox_inches='tight')
print("Saved fig_correlations.png")
plt.show()


Haddara (n=70): mean |r| = 0.553
Maniscalco (n=22): mean |r| = 0.627


Shekhar (n=20): mean |r| = 0.657


Saved fig_correlations.png
